#### Model 2 Building
(CLINICAL PCOS MODEL)

In [1]:
import pandas as pd
import numpy as np
import mlflow

/opt/anaconda3/envs/tf_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df2=pd.read_csv("Feature Engnineered dataset.csv")
df2.head()

,pcos,age,weight,height,bmi,had_abortion,blood_group,cycle_type,marriage_years,pregnant,...,amh,prolactin,vitamin_d3,progesterone,random_blood_sugar,bp_systolic,bp_diastolic,avg_follicle_size,total_follicles,endometrium_thickness
0,0,28,44.6,1.520,19.304017,0,O+,Regular,7.0,0,...,2.07,45.16,17.1,0.57,92.0,110,80,18.0,6,8.5
1,0,36,65.0,1.615,24.921163,0,O+,Regular,11.0,1,...,1.53,20.09,61.3,0.97,92.0,120,70,14.5,8,3.7
2,1,33,68.8,1.650,25.270891,0,A+,Regular,10.0,1,...,6.63,10.52,49.7,0.36,84.0,120,80,19.0,28,10.0
3,0,37,65.0,1.480,29.674945,0,B+,Regular,4.0,0,...,1.22,36.90,33.4,0.36,76.0,120,70,14.5,4,7.5
4,0,25,52.0,1.610,20.060954,0,A+,Regular,1.0,1,...,2.26,30.09,43.8,0.38,84.0,120,80,15.0,7,7.0


In [3]:
df2.columns

Index(['pcos', 'age', 'weight', 'height', 'bmi', 'had_abortion', 'blood_group',
       'cycle_type', 'marriage_years', 'pregnant', 'weight_gain',
       'hair_growth', 'skin_darkening', 'hair_loss', 'pimples', 'fast_food',
       'regular_exercise', 'pulse_rate', 'respiratory_rate', 'hemoglobin',
       'fsh', 'lh', 'lh_fsh_ratio', 'waist_hip_ratio', 'tsh', 'amh',
       'prolactin', 'vitamin_d3', 'progesterone', 'random_blood_sugar',
       'bp_systolic', 'bp_diastolic', 'avg_follicle_size', 'total_follicles',
       'endometrium_thickness'],
      dtype='object')

In [4]:
df2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541 entries, 0 to 540
Data columns (total 35 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   pcos                   541 non-null    int64  
 1   age                    541 non-null    int64  
 2   weight                 541 non-null    float64
 3   height                 541 non-null    float64
 4   bmi                    541 non-null    float64
 5   had_abortion           541 non-null    int64  
 6   blood_group            541 non-null    object 
 7   cycle_type             541 non-null    object 
 8   marriage_years         540 non-null    float64
 9   pregnant               541 non-null    int64  
 10  weight_gain            541 non-null    int64  
 11  hair_growth            541 non-null    int64  
 12  skin_darkening         541 non-null    int64  
 13  hair_loss              541 non-null    int64  
 14  pimples                541 non-null    int64  
 15  fast_f

In [5]:
df2.isna().sum()

pcos                     0
age                      0
weight                   0
height                   0
bmi                      0
had_abortion             0
blood_group              0
cycle_type               0
marriage_years           1
pregnant                 0
weight_gain              0
hair_growth              0
skin_darkening           0
hair_loss                0
pimples                  0
fast_food                1
regular_exercise         0
pulse_rate               0
respiratory_rate         0
hemoglobin               0
fsh                      0
lh                       0
lh_fsh_ratio             0
waist_hip_ratio          0
tsh                      0
amh                      1
prolactin                0
vitamin_d3               0
progesterone             0
random_blood_sugar       0
bp_systolic              0
bp_diastolic             0
avg_follicle_size        0
total_follicles          0
endometrium_thickness    0
dtype: int64

In [6]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(df2.drop('pcos',axis=1),df2['pcos'],test_size=0.25,random_state=42,stratify=df2['pcos'])

In [7]:
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import VotingClassifier
from xgboost import XGBClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

In [8]:
### dividing columns based on preprocessing techniques we want to apply on them
numerical=['age', 'weight', 'height', 'bmi', 'had_abortion', 'marriage_years', 'pregnant', 'weight_gain',
       'hair_growth', 'skin_darkening', 'hair_loss', 'pimples', 'fast_food',
       'regular_exercise', 'pulse_rate', 'respiratory_rate', 'hemoglobin',
       'fsh', 'lh', 'lh_fsh_ratio', 'waist_hip_ratio', 'tsh', 'amh',
       'prolactin', 'vitamin_d3', 'progesterone', 'random_blood_sugar',
       'bp_systolic', 'bp_diastolic', 'avg_follicle_size', 'total_follicles',
       'endometrium_thickness']
ordinal=['cycle_type']
nominal=['blood_group']


In [9]:
from sklearn.pipeline import Pipeline
numerical_pipe = Pipeline(steps=[
    ('imputer', SimpleImputer()),
    ('scaler', StandardScaler())
])

In [10]:
transformer=ColumnTransformer(transformers=[
    ('tnf1',numerical_pipe,numerical),
    ('tnf2',OneHotEncoder(drop='first',handle_unknown='ignore'),nominal),
    ('tnf3',OrdinalEncoder(categories=[['Regular','Irregular']]),ordinal)],
                              remainder='drop')
x_train_trans=transformer.fit_transform(x_train)
x_test_trans=transformer.transform(x_test)

In [11]:
### training differennt models
### handling imbalanced data by computing class weight and assigning those class weight in the model
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes = np.unique(y_train)
weights = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=y_train
)

class_weight = dict(zip(classes, weights))


### calculating scale_pos_weights for XGBoost this the type of class weight used in xgboost
scale_pos_weight = 364 / 177   #where 364=no.of 0's, 177=no.of 1's

In [23]:
mlflow.set_tracking_uri("sqlite:////Users/atharvashinde/Desktop/Pcos api/mlflow.db")

mlflow.set_experiment("Pcos-model2 training")

2026/06/04 13:16:46 INFO mlflow.tracking.fluent: Experiment with name 'Pcos-model2 training' does not exist. Creating a new experiment.


<Experiment: artifact_location='/Users/atharvashinde/mlruns/2', creation_time=1780559206978, effective_trace_archival_retention=None, experiment_id='2', last_update_time=1780559206978, lifecycle_stage='active', name='Pcos-model2 training', tags={}, trace_location=None, workspace='default'>

In [24]:
### model 1= Random Forest
from sklearn.metrics import classification_report
mlflow.autolog()
with mlflow.start_run(run_name='RandomForest'):
    model_forest=RandomForestClassifier(class_weight=class_weight)
    model_forest.fit(x_train_trans,y_train)
    y_pred_1=model_forest.predict(x_test_trans)
    mlflow.log_param('model2_name',"RandomForest")
    report1=classification_report(y_test,y_pred_1,output_dict=True)
    mlflow.log_metric("recall_class_0",report1['0']['recall'])
    mlflow.log_metric("recall_class_1",report1['1']['recall'])
    mlflow.log_metric("precision_class_0",report1['0']['precision'])
    mlflow.log_metric("precision_class_1",report1['1']['precision'])
    
    

2026/06/04 13:16:50 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.
2026/06/04 13:16:50 INFO mlflow.tracking.fluent: Autologging successfully enabled for xgboost.
2026/06/04 13:16:50 WARNING mlflow.sklearn: Failed to log training dataset information to MLflow Tracking. Reason: 'Series' object has no attribute 'flatten'
2026/06/04 13:16:50 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


In [25]:
### model 2= SVC
mlflow.autolog()
with mlflow.start_run(run_name='SVC'):
    
    model_svc=SVC(class_weight=class_weight)
    model_svc.fit(x_train_trans,y_train)
    y_pred_2=model_svc.predict(x_test_trans)
    mlflow.log_param('model2_name',"SVC")
    report2=classification_report(y_test,y_pred_2,output_dict=True)
    mlflow.log_metric("recall_class_0",report2['0']['recall'])
    mlflow.log_metric("recall_class_1",report2['1']['recall'])
    mlflow.log_metric("precision_class_0",report2['0']['precision'])
    mlflow.log_metric("precision_class_1",report2['1']['precision'])
    
    

2026/06/04 13:16:53 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.
2026/06/04 13:16:53 INFO mlflow.tracking.fluent: Autologging successfully enabled for xgboost.
2026/06/04 13:16:53 WARNING mlflow.sklearn: Failed to log training dataset information to MLflow Tracking. Reason: 'Series' object has no attribute 'flatten'
2026/06/04 13:16:53 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


In [26]:
### model 3 = XGBOOST
mlflow.autolog()
with mlflow.start_run(run_name='XGB'):
    
    model_xg=XGBClassifier(scale_pos_weight=scale_pos_weight)
    model_xg.fit(x_train_trans,y_train)
    y_pred_3=model_xg.predict(x_test_trans)
    mlflow.log_param('model2_name',"XGB")
    report3=classification_report(y_test,y_pred_3,output_dict=True)
    mlflow.log_metric("recall_class_0",report3['0']['recall'])
    mlflow.log_metric("recall_class_1",report3['1']['recall'])
    mlflow.log_metric("precision_class_0",report3['0']['precision'])
    mlflow.log_metric("precision_class_1",report3['1']['precision'])
    
    

2026/06/04 13:16:56 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.
2026/06/04 13:16:56 INFO mlflow.tracking.fluent: Autologging successfully enabled for xgboost.
2026/06/04 13:16:56 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


In [27]:
from sklearn.metrics import classification_report
print("Random Forest :") 
print(classification_report(y_test,y_pred_1))
print("SVC :")
print(classification_report(y_test,y_pred_2))
print("XGBClassifier :")
print(classification_report(y_test,y_pred_3))


Random Forest :
              precision    recall  f1-score   support

           0       0.91      0.97      0.94        92
           1       0.92      0.80      0.85        44

    accuracy                           0.91       136
   macro avg       0.91      0.88      0.90       136
weighted avg       0.91      0.91      0.91       136

SVC :
              precision    recall  f1-score   support

           0       0.91      0.93      0.92        92
           1       0.86      0.82      0.84        44

    accuracy                           0.90       136
   macro avg       0.89      0.88      0.88       136
weighted avg       0.90      0.90      0.90       136

XGBClassifier :
              precision    recall  f1-score   support

           0       0.94      0.95      0.94        92
           1       0.88      0.86      0.87        44

    accuracy                           0.92       136
   macro avg       0.91      0.90      0.91       136
weighted avg       0.92      0.92   

In [28]:
#### svc and xgb are performing equally well lets see how they perform if we do hyperparameter tuning
### FOR SVC
from sklearn.model_selection import RandomizedSearchCV,StratifiedKFold
from sklearn.metrics import recall_score,make_scorer
param_grid = {'kernel':['linear', 'poly', 'rbf', 'sigmoid'],
              'C':[0.1, 1, 10, 100],
              'gamma':['scale', 'auto'],
              'class_weight':[None,'balanced']}
recall_pos=make_scorer(recall_score,pos_label=1)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

gv=RandomizedSearchCV(estimator=model_svc,param_distributions=param_grid,cv=cv,scoring=recall_pos,n_jobs=-1)
gv.fit(x_train_trans,y_train)
print("Best params:",gv.best_params_)
print("best recall score for svc after hyperparameterb tuning:",gv.best_score_)

2026/06/04 13:17:01 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '36b07a82d6f24657bc8258abea93459a', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
2026/06/04 13:17:01 WARNING mlflow.sklearn: Failed to log training dataset information to MLflow Tracking. Reason: 'Series' object has no attribute 'flatten'
2026/06/04 13:17:05 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/06/04 13:17:07 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object

Best params: {'kernel': 'linear', 'gamma': 'scale', 'class_weight': 'balanced', 'C': 0.1}
best recall score for svc after hyperparameterb tuning: 0.8706552706552706


In [29]:
### FOR XGBOOST
param_grid_xgb = {
    "n_estimators": [100, 200, 300],
    "learning_rate": [0.01, 0.05, 0.1, 0.2],
    "max_depth": [3, 4, 5, 6],
    "subsample": [0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.7, 0.8, 0.9, 1.0],
    "gamma": [0, 0.1, 0.2, 0.5],
    "scale_pos_weight": [1, 2, 3]
}
recall_pos = make_scorer(recall_score, pos_label=1)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

gv2 = RandomizedSearchCV(estimator=model_xg,param_distributions=param_grid_xgb,scoring=recall_pos,cv=cv,n_jobs=-1,verbose=1
)

gv2.fit(x_train_trans, y_train)



2026/06/04 13:17:12 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '0f301d5f5d044c13bbcf0f25bf660b7e', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
2026/06/04 13:17:12 WARNING mlflow.sklearn: Failed to log training dataset information to MLflow Tracking. Reason: 'Series' object has no attribute 'flatten'


Fitting 5 folds for each of 10 candidates, totalling 50 fits


2026/06/04 13:17:13 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/06/04 13:17:16 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/06/04 13:17:18 INFO mlflow.sklearn.utils: Logging the 5 best runs, 5 runs will be omitted.
2026/06/04 13:17:18 WARNING mlflow.sklearn: Encountered exception during creation of child runs for pa

,estimator,"XGBClassifier...ree=None, ...)"
,param_distributions,"{'colsample_bytree': [0.7, 0.8, ...], 'gamma': [0, 0.1, ...], 'learning_rate': [0.01, 0.05, ...], 'max_depth': [3, 4, ...], ...}"
,n_iter,10
,scoring,"make_scorer(r..., pos_label=1)"
,n_jobs,-1
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,1
,pre_dispatch,'2*n_jobs'
,random_state,None
,error_score,nan


In [30]:
print(gv2.best_params_)

{'subsample': 0.7, 'scale_pos_weight': 3, 'n_estimators': 100, 'max_depth': 3, 'learning_rate': 0.01, 'gamma': 0.5, 'colsample_bytree': 0.7}


In [31]:
print(gv2.best_score_)

0.8863247863247864


In [32]:
#### xgboost performed better
mlflow.autolog()
with mlflow.start_run(run_name='XGB_hyperparametr_tuned'):
    
    final_model=gv2.best_estimator_
    final_model.fit(x_train_trans,y_train)
    y_pred_final=final_model.predict(x_test_trans)
    mlflow.log_param('model2_name',"XGB_hyperparametr_tuned")
    report4=classification_report(y_test,y_pred_final,output_dict=True)
    mlflow.log_metric("recall_class_0",report4['0']['recall'])
    mlflow.log_metric("recall_class_1",report4['1']['recall'])
    mlflow.log_metric("precision_class_0",report4['0']['precision'])
    mlflow.log_metric("precision_class_1",report4['1']['precision'])
    
    

2026/06/04 13:17:22 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.
2026/06/04 13:17:22 INFO mlflow.tracking.fluent: Autologging successfully enabled for xgboost.
2026/06/04 13:17:23 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


In [33]:
## classification report
print(classification_report(y_test,y_pred_final))

              precision    recall  f1-score   support

           0       0.94      0.86      0.90        92
           1       0.75      0.89      0.81        44

    accuracy                           0.87       136
   macro avg       0.85      0.87      0.86       136
weighted avg       0.88      0.87      0.87       136



In [34]:
### lets see our final_model is overfitting or not
from sklearn.metrics import recall_score

# Train predictions
y_train_pred = final_model.predict(x_train_trans)
train_recall = recall_score(y_train, y_train_pred)

# Test predictions
y_test_pred = final_model.predict(x_test_trans)
test_recall = recall_score(y_test, y_test_pred)

print("Train Recall:", train_recall)
print("Test Recall:", test_recall)

Train Recall: 0.9849624060150376
Test Recall: 0.8863636363636364


In [35]:
import pickle
bundle={"transformer":transformer,
        "model":final_model}
model_path="model_2.pkl"
with open(model_path,"wb") as f:
    pickle.dump(bundle,f)

In [37]:
loaded_model=mlflow.xgboost.load_model(f"models:/Model_2_XGB/1")

In [39]:
loaded_model.predict(x_test_trans)

array([0, 0, 1, 1, 0, 0, 0, 1, 1, 0, 1, 1, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0,
       0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 1,
       0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 1, 1, 0, 1, 0,
       0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0,
       0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1,
       0, 0, 0, 1])